Carregar o CSV e imprimir shape e columns

In [1]:
!pip install pandas


zsh:1: command not found: pip


In [2]:
import pandas as pd
import numpy 



In [3]:
df = pd.read_csv("data/com_texto.csv")
print("Formato:", df.shape)
print("Colunas:", df.columns)

Formato: (1882, 7)
Colunas: Index(['Link', 'Título da checagem', 'Data da checagem', 'Natureza da notícia',
       'Candidato(s) favorecidos(s) pela notícia falsa', 'Agência', 'texto'],
      dtype='str')


In [4]:
df["Data da checagem"] = pd.to_datetime(df["Data da checagem"], format="%m/%d/%Y", errors="coerce")
df["Data da checagem"].isna().sum()

np.int64(8)

In [5]:
print("Data mínima:", df["Data da checagem"].min())
print("Data máxima:", df["Data da checagem"].max())

Data mínima: 2022-08-01 00:00:00
Data máxima: 2022-12-01 00:00:00


In [6]:
df["Agência"].value_counts()

Agência
Boatos.org                           315
Aos Fatos                            314
Agência Lupa                         245
UOL Confere                          228
AFP Checamos                         196
Fato ou Boato (Justiça Eleitoral)    184
Projeto Comprova                     175
Fato ou Fake                         172
E-farsas                              50
Folha de S. Paulo                      2
CNJ                                    1
Name: count, dtype: int64

In [7]:
df["Natureza da notícia"].value_counts()

Natureza da notícia
Falsa                      1820
Verdadeira                    9
Parcialmente verdadeira       3
Name: count, dtype: int64

In [8]:
df["Título da checagem"].sample(10, random_state=42)

1519    É falso áudio em que Bolsonaro xinga Michelle ...
1247    É falso que TSE negou pedido de candidatura à ...
798     Forças Armadas podem intervir a qualquer momen...
756     Ministério da Defesa fez nota oficial que cita...
710     Confirmada a fraude: FFAA deve apresentar rela...
414     ‘O Globo’ não teve acesso antecipado a resulta...
120     Sigilos de cem anos não são estabelecidos por ...
1347    É falso que pesquisa de boca de urna mostra Bo...
453     Vídeo que mostra Malafaia em Brasília não é at...
1814    TV Globo não gravou especial de fim de ano com...
Name: Título da checagem, dtype: str

In [9]:
df["texto"].head(2).str[:300]

0    O vídeo do suposto saque em Tucumán, na Argent...
1    Os homens com tatuagens nazistas fotografados ...
Name: texto, dtype: str

In [10]:
df["Título da checagem"].duplicated().sum()

np.int64(20)

In [11]:
df[df.duplicated("Título da checagem", keep=False)].sort_values("Título da checagem")[["Título da checagem", "Agência", "Data da checagem"]]

,Título da checagem,Agência,Data da checagem
1623,Artigo 142 não prevê intervenção militar nem f...,Projeto Comprova,2022-11-08
827,Artigo 142 não prevê intervenção militar nem f...,Folha de S. Paulo,2022-11-09
969,Boletins de urna mostrados em vídeo não são de...,Fato ou Boato (Justiça Eleitoral),2022-10-13
825,Boletins de urna mostrados em vídeo não são de...,Aos Fatos,2022-10-10
369,Boletins de urna mostrados em vídeo não são de...,Aos Fatos,2022-10-10
74,Código Eleitoral prevê prisão em diversos caso...,AFP Checamos,2022-09-23
900,Código Eleitoral prevê prisão em diversos caso...,Fato ou Boato (Justiça Eleitoral),2022-09-24
888,Lei eleitoral proíbe assinatura com número de ...,Fato ou Boato (Justiça Eleitoral),2022-08-31
251,Lei eleitoral proíbe assinatura com número de ...,Aos Fatos,2022-08-26
1736,Vini Jr não criticou Bolsonaro no Twitter; pos...,UOL Confere,2022-10-05


Limpeza dos duplicados: remove só o tipo A (mesmo título + mesma agência + mesma data = erro de coleta). Títulos iguais entre agências diferentes são cobertura paralela ou republicação e ficam no corpus. A flag `agregador` marca o Fato ou Boato (TSE), que republica checagens de parceiros.

In [12]:
antes = len(df)
df = df.drop_duplicates(subset=["Título da checagem", "Agência", "Data da checagem"]).reset_index(drop=True)
print(f"Removidas: {antes - len(df)} linhas -> Formato: {df.shape}")

df["agregador"] = df["Agência"].str.contains("Fato ou Boato")
print("Linhas de agregador:", df["agregador"].sum())

# Títulos que ainda se repetem = mesma checagem em agências diferentes (tipos B e C)
print("Títulos ainda repetidos:", df["Título da checagem"].duplicated().sum())

Removidas: 5 linhas -> Formato: (1877, 7)
Linhas de agregador: 184
Títulos ainda repetidos: 15


In [13]:
import re

# Enquadramento de veredito no INÍCIO do título ("É falso que", "É #FAKE", "Vídeo engana ao afirmar que"...)
PREFIX = re.compile(
    r"^\s*(#?checamos:?\s*)?"
    r"(é\s+(falso|enganoso|verdade|exagerado|impreciso|montagem|mentira|#fake)(\s+que)?|"
    r"não\s+é\s+verdade\s+que|falso:|enganoso:|verdadeiro:|"
    r"(vídeo|mulher|posts?|foto|imagem)\s+(engana|desinforma)\s+ao\s+(afirmar|repetir|dizer)?\s*(que)?)\s*",
    flags=re.IGNORECASE)

# Marca de veredito no FINAL do título (Boatos.org: "... #boato")
SUFFIX = re.compile(r"\s*#boato\s*$", flags=re.IGNORECASE)

df["claim"] = (df["Título da checagem"]
               .str.replace(PREFIX, "", regex=True)
               .str.replace(SUFFIX, "", regex=True)
               .str.strip()
               # maiúscula só na 1ª letra; capitalize() deixaria o resto minúsculo e estragaria nomes próprios
               .str.replace(r"^(\w)", lambda m: m.group(1).upper(), regex=True))

# "Veja o que é #FATO ou #FAKE na entrevista de X" = checagem de várias alegações de uma vez, não uma alegação
df["multi_claim"] = df["Título da checagem"].str.match(r"veja o que", case=False)
print("Checagens multi-alegação:", df["multi_claim"].sum())

df[df["Título da checagem"] == df["claim"]].sample(15, random_state=42)[["Título da checagem"]]

Checagens multi-alegação: 19


,Título da checagem
1760,Lula não parabenizou recepção de Tarcísio em P...
1256,É falsa entrevista de Luciano Hang sobre inves...
1366,G1 não noticiou que Bolsonaro confirmou Collor...
945,Urna que não permitia votação para presidente ...
452,Vídeo não mostra ministro da Defesa dizendo qu...
1317,É de 2018 vídeo de garis entregando urnas elet...
1318,É antigo vídeo em que prédio pega fogo durante...
965,Boletins de urna mostrados em vídeo não são de...
390,"No Flow, Lula infla legado do PT e distorce fa..."
72,Foto de Lula com a rainha Elizabeth II não foi...


In [14]:
(df["Título da checagem"]!=df["claim"]).sum()

np.int64(872)

In [15]:
df.to_csv("data/com_texto_limpo.csv", index=False)